# Statistical intelligence
Short monthly series require cautious interpretation. Statistical models below describe association and model uncertainty, not randomized causal effects.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
root=Path.cwd()
if root.name=='notebooks': root=root.parent
sys.path.insert(0,str(root))
from src.statistical_analysis import describe_series, calendar_series
from src.trend_analysis import estimate_trend
from src.cost_diagnostics import shapley_cost
def table(name): return pd.read_parquet(root / f'outputs/phase3/tables/{name}.parquet')
def chart(name): display(Image(filename=str(root / f'outputs/phase3/charts/{name}.png')))
national=pd.read_parquet(root/'outputs/tables/national_monthly_trends.parquet')


## Trend methods
OLS uses elapsed calendar month. HAC covariance uses Bartlett lags 3, a small-sample correction and t intervals. Theil–Sen uses the median paired slope; Kendall tau is descriptive without iid p-values. HAC 1/6/12 and month indicators are sensitivities.

In [ ]:
display(table('trend_summary'))
display(estimate_trend(calendar_series(national,'prescription_items')))
chart('16_trend_sensitivity')

## Seasonal evidence
Only three annual cycles are available. Annual-mean indices retain within-year trend. Calendar-day normalization is a sensitivity, not working-day adjustment. Robust STL period 12 displays a seasonal component without establishing stable seasonality.

In [ ]:
display(table('seasonality_summary'))
display(table('monthly_seasonal_indices').head(12))
chart('08_monthly_seasonal_profile')
chart('10_autocorrelation')

## Exact cost identity
$C=V\sum_j s_jp_j$. Shapley contributions average all six orders for changing volume $V$, shares $s$, and within-presentation NIC/item $p$. They reconcile exactly; the factor allocation is a convention, not a uniquely identified causal mechanism.

In [ ]:
display(table('cost_decomposition_diagnostics'))
display(shapley_cost(100,[.6,.4],[2,5],120,[.5,.5],[1,4]))
chart('11_nic_decomposition')

## Anomalies and robustness
MAD z-scores use 0.67449 normalization within each series. Zero MAD and histories below 12 observations give undefined scores. Original IQR flags are retained. Classifications are heuristic review labels. Exclusions affect diagnostic copies only.

In [ ]:
display(table('extreme_value_review_phase3').groupby('classification').size())
display(table('anomaly_diagnostics').head(12))
display(table('robustness_summary'))
chart('14_anomaly_timeline')

## Forecasting implications
No forecast model or backtest is trained. Start with naive, seasonal-naive and drift baselines, then damped ETS and regularized lag regression. Evaluate 1–3 month horizons with chronological origins; refit preprocessing and eligibility inside folds.

In [ ]:
display(table('forecasting_diagnostics').query("level == 'national'"))
chart('18_forecasting_readiness')
assert table('cost_decomposition_diagnostics').reconciliation_residual_gbp.abs().max()<.01